# Wanderbricks Reviews Analysis

**Dataset:** `samples.wanderbricks.reviews`

**Difficulty:** Medium

**Topics:** filter, aggregation, window functions, conditional

In [0]:
from pyspark.sql import functions as F, types as T
from pyspark.sql import Window as W

reviews = spark.read.table("samples.wanderbricks.reviews")

## Problem 1

Calculate average rating per property. Only include non-deleted reviews (`is_deleted = false`).
Sort by `avg_rating` descending.

**Expected output columns:**
- `property_id`
- `avg_rating`
- `review_count`

In [0]:
# Problem 1 - write your solution here
# Assign your result to: result_1

result_1 = reviews.filter(
    ~F.col("is_deleted")
).groupBy("property_id").agg(
    F.avg("rating").alias("avg_rating"),
    F.count("review_id").alias("review_count")
).orderBy(F.col("avg_rating").desc())

result_1.display()

In [0]:
# ── Tests for Problem 1 ──────────────────────────────────────────
assert result_1 is not None, "result_1 is None - did you assign your DataFrame?"
assert hasattr(result_1, 'columns'), "result_1 must be a Spark DataFrame"
cols = [c.lower() for c in result_1.columns]
assert 'property_id' in cols, "Missing column: property_id"
assert 'avg_rating' in cols, "Missing column: avg_rating"
assert 'review_count' in cols, "Missing column: review_count"
assert len(cols) == 3, f"Expected exactly 3 columns, got {len(cols)}: {cols}"
cnt = result_1.count()
assert cnt > 0, f"Expected rows > 0, got {cnt}"
max_rating = result_1.agg(F.max('avg_rating')).collect()[0][0]
assert max_rating <= 5.0, f"Expected avg_rating <= 5.0, got max={max_rating}"
print(f"Problem 1 passed ✓  ({cnt} rows)")

## Problem 2

Find properties with a perfect average rating of 5.0, using only non-deleted reviews.

**Expected output columns:**
- `property_id`
- `avg_rating`
- `review_count`

In [0]:
# Problem 2 - write your solution here
# Assign your result to: result_2

result_2 = (
    reviews
    .filter(~F.col("is_deleted"))
    .groupBy("property_id")
    .agg(
        F.avg("rating").alias("avg_rating"),
        F.count("review_id").alias("review_count")
    )
    .filter(F.col("avg_rating") == 5.0)
)

result_2.display()

In [0]:
# ── Tests for Problem 2 ──────────────────────────────────────────
assert result_2 is not None, "result_2 is None - did you assign your DataFrame?"
assert hasattr(result_2, 'columns'), "result_2 must be a Spark DataFrame"
cols = [c.lower() for c in result_2.columns]
assert 'property_id' in cols, "Missing column: property_id"
assert 'avg_rating' in cols, "Missing column: avg_rating"
assert 'review_count' in cols, "Missing column: review_count"
assert len(cols) == 3, f"Expected exactly 3 columns, got {len(cols)}: {cols}"
cnt = result_2.count()
assert cnt >= 0, f"Expected rows >= 0, got {cnt}"
if cnt > 0:
    min_rating = result_2.agg(F.min('avg_rating')).collect()[0][0]
    assert min_rating == 5.0, f"Expected all avg_rating == 5.0, found min={min_rating}"
print(f"Problem 2 passed ✓  ({cnt} rows)")

## Problem 3

Show rating distribution: count of reviews per rating value and the percentage each rating represents.

**Expected output columns:**
- `rating`
- `count`
- `percentage`

In [0]:
# Problem 3 - write your solution here
# Assign your result to: result_3
w = W.rowsBetween(W.unboundedPreceding, W.unboundedFollowing)
result_3 = reviews.filter(~F.col("is_deleted")).groupBy("rating").count().withColumn(
    "percentage",
    F.round(100 * F.col("count") / F.sum("count").over(w), 2)
).orderBy(F.col("rating").desc())

result_3.display()

In [0]:
# ── Tests for Problem 3 ──────────────────────────────────────────
assert result_3 is not None, "result_3 is None - did you assign your DataFrame?"
assert hasattr(result_3, 'columns'), "result_3 must be a Spark DataFrame"
cols = [c.lower() for c in result_3.columns]
assert 'rating' in cols, "Missing column: rating"
assert 'count' in cols, "Missing column: count"
assert 'percentage' in cols, "Missing column: percentage"
assert len(cols) == 3, f"Expected exactly 3 columns, got {len(cols)}: {cols}"
cnt = result_3.count()
assert cnt > 0, f"Expected rows > 0, got {cnt}"
max_pct = result_3.agg(F.max('percentage')).collect()[0][0]
min_pct = result_3.agg(F.min('percentage')).collect()[0][0]
assert max_pct <= 100, f"Expected percentage <= 100, got max={max_pct}"
assert min_pct >= 0, f"Expected percentage >= 0, got min={min_pct}"
print(f"Problem 3 passed ✓  ({cnt} rows)")

## Problem 4

Count reviews per month by extracting year and month from `created_at`.

**Expected output columns:**
- `year`
- `month`
- `review_count`

In [0]:
# Problem 4 - write your solution here
# Assign your result to: result_4

result_4 = reviews.filter(~F.col("is_deleted")).groupBy(
    F.year("created_at").alias("year"),
    F.month("created_at").alias("month")
).agg(
    F.count("review_id").alias("review_count")
).orderBy("year", "month")

result_4.display()

In [0]:
# ── Tests for Problem 4 ──────────────────────────────────────────
assert result_4 is not None, "result_4 is None - did you assign your DataFrame?"
assert hasattr(result_4, 'columns'), "result_4 must be a Spark DataFrame"
cols = [c.lower() for c in result_4.columns]
assert 'year' in cols, "Missing column: year"
assert 'month' in cols, "Missing column: month"
assert 'review_count' in cols, "Missing column: review_count"
assert len(cols) == 3, f"Expected exactly 3 columns, got {len(cols)}: {cols}"
cnt = result_4.count()
assert cnt > 0, f"Expected rows > 0, got {cnt}"
max_month = result_4.agg(F.max('month')).collect()[0][0]
assert max_month <= 12, f"Expected month <= 12, got max={max_month}"
print(f"Problem 4 passed ✓  ({cnt} rows)")

## Problem 5

Find the top 10 properties by number of non-deleted reviews, including their average rating.

**Expected output columns:**
- `property_id`
- `review_count`
- `avg_rating`

In [0]:
# Problem 5 - write your solution here
# Assign your result to: result_5

result_5 = (
    reviews
    .filter(~F.col("is_deleted"))
    .groupBy("property_id")
    .agg(
        F.count("review_id").alias("review_count"),
        F.avg("rating").alias("avg_rating")
    )
    .orderBy(F.col("review_count").desc())
    .limit(10)
)

result_5.display()

In [0]:
# ── Tests for Problem 5 ──────────────────────────────────────────
assert result_5 is not None, "result_5 is None - did you assign your DataFrame?"
assert hasattr(result_5, 'columns'), "result_5 must be a Spark DataFrame"
cols = [c.lower() for c in result_5.columns]
assert 'property_id' in cols, "Missing column: property_id"
assert 'review_count' in cols, "Missing column: review_count"
assert 'avg_rating' in cols, "Missing column: avg_rating"
assert len(cols) == 3, f"Expected exactly 3 columns, got {len(cols)}: {cols}"
cnt = result_5.count()
assert cnt > 0, f"Expected rows > 0, got {cnt}"
assert cnt <= 10, f"Expected at most 10 rows (top 10), got {cnt}"
print(f"Problem 5 passed ✓  ({cnt} rows)")

## Problem 6

Using a window function, compute a running average rating per property ordered by `created_at`.

**Expected output columns:**
- `review_id`
- `property_id`
- `rating`
- `created_at`
- `running_avg_rating`

In [0]:
# Problem 6 - write your solution here
# Assign your result to: result_6
w = W.partitionBy("property_id").orderBy("created_at").rowsBetween(W.unboundedPreceding, W.currentRow)
result_6 = (
    reviews
    .select(
        "review_id",
        "property_id",
        "rating",
        "created_at",
        F.avg("rating").over(w).alias("running_avg_rating")
    )
    .orderBy("property_id", "created_at")
)

result_6.display()

In [0]:
# ── Tests for Problem 6 ──────────────────────────────────────────
assert result_6 is not None, "result_6 is None - did you assign your DataFrame?"
assert hasattr(result_6, 'columns'), "result_6 must be a Spark DataFrame"
cols = [c.lower() for c in result_6.columns]
assert 'review_id' in cols, "Missing column: review_id"
assert 'property_id' in cols, "Missing column: property_id"
assert 'rating' in cols, "Missing column: rating"
assert 'created_at' in cols, "Missing column: created_at"
assert 'running_avg_rating' in cols, "Missing column: running_avg_rating"
assert len(cols) == 5, f"Expected exactly 5 columns, got {len(cols)}: {cols}"
cnt = result_6.count()
assert cnt > 0, f"Expected rows > 0, got {cnt}"
max_avg = result_6.agg(F.max('running_avg_rating')).collect()[0][0]
assert max_avg <= 5.0, f"Expected running_avg_rating <= 5.0, got max={max_avg}"
min_avg = result_6.agg(F.min('running_avg_rating')).collect()[0][0]
assert min_avg >= 0, f"Expected running_avg_rating >= 0, got min={min_avg}"
print(f"Problem 6 passed ✓  ({cnt} rows)")

## Problem 7

Find properties where the most recent review (max `created_at`) has a rating below 3.0.

**Expected output columns:**
- `property_id`
- `latest_review_date`
- `latest_rating`

In [0]:
# Problem 7 - write your solution here
# Assign your result to: result_7
w = W.partitionBy("property_id").orderBy(F.col("created_at").desc())
result_7 = reviews.filter(~F.col("is_deleted")).withColumn(
    "rn",
    F.row_number().over(w)
).filter(
    (F.col("rn") == 1) & (F.col("rating") < 3.0)
).select(
    "property_id",
    F.to_date("created_at").alias("latest_review_date"),
    F.col("rating").alias("latest_rating")
)

result_7.display()

In [0]:
# ── Tests for Problem 7 ──────────────────────────────────────────
assert result_7 is not None, "result_7 is None - did you assign your DataFrame?"
assert hasattr(result_7, 'columns'), "result_7 must be a Spark DataFrame"
cols = [c.lower() for c in result_7.columns]
assert 'property_id' in cols, "Missing column: property_id"
assert 'latest_review_date' in cols, "Missing column: latest_review_date"
assert 'latest_rating' in cols, "Missing column: latest_rating"
assert len(cols) == 3, f"Expected exactly 3 columns, got {len(cols)}: {cols}"
cnt = result_7.count()
assert cnt >= 0, f"Expected rows >= 0, got {cnt}"
if cnt > 0:
    max_rating = result_7.agg(F.max('latest_rating')).collect()[0][0]
    assert max_rating < 3.0, f"Expected latest_rating < 3.0 for all rows, got max={max_rating}"
print(f"Problem 7 passed ✓  ({cnt} rows)")